# Etapa 2: Conversión a Capa Bronze (CSV a Parquet)

Este cuaderno realiza la conversión directa 1:1 de los archivos CSV de la capa **Raw** a formato columnar estructurado **Parquet** en la capa **Bronze**, mejorando la eficiencia de almacenamiento y lectura analítica.

In [ ]:
import sys
import os
from pathlib import Path

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")
    
    # Intentar montar Google Drive automáticamente si no está montado
    if not Path('/content/drive').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print("No se pudo montar Drive automáticamente. Por favor, móntelo en el panel izquierdo de Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # Rutas de búsqueda comunes en Google Drive y Colab
    possible_paths = [
        Path('/content/drive/MyDrive/Colab Notebooks/Proyecto'),
        Path('/content/drive/MyDrive/Proyecto'),
        Path('/content/Proyecto'),
        Path('/content')
    ]
    for p in possible_paths:
        if (p / "notebooks").exists():
            current_dir = p / "notebooks"
            break

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Ejecutar Ingesta a Capa Bronze
Importamos y ejecutamos la lógica de `DataProcessor` para realizar la ingesta columnar.

In [ ]:
from src.data_processing import DataProcessor

processor = DataProcessor(PROJECT_ROOT)
processor.raw_to_bronze()

## 2. Validación de Archivos Parquet Generados

In [ ]:
bronze_dir = PROJECT_ROOT / "data" / "bronze"
parquet_files = list(bronze_dir.glob("*.parquet"))

print(f"Archivos Parquet creados en Capa Bronze: {len(parquet_files)}")
for p in parquet_files:
    print(f"✔ {p.name} | Tamaño: {p.stat().st_size / 1024:.2f} KB")